- TO DO:
    - mini-batches learning process
    - stochastic gradient descent
    - Currently, the mlp archtecture is specialized to the MNIST classification problem. I need to generalize each learning step for any data set multiclassification problem.
    - Vizualization (learning and inference process).


- Open questions:
    - How to find the best number of layers and nodes?
    - How to find the best combination of activation functions?
    - If I am working with mini-batches, is there a optimal batch size? If yes, how to computate?
    - Did I shuffle my data? Was it necessary? Justify.

In [29]:
import pandas as pd
import numpy as np
import idx2numpy

In [8]:
# read data
X_train = idx2numpy.convert_from_file('data/mnist/train-images.idx3-ubyte')
y_train = idx2numpy.convert_from_file('data/mnist/train-labels.idx1-ubyte')

X_train = (X_train.reshape(X_train.shape[0], -1) / 255.0).T
y_train = y_train.reshape(1, -1)

print("shape:")
print("X_train -", X_train.shape)
print("y_train -", y_train.shape)

shape:
X_train - (784, 60000)
y_train - (1, 60000)


In [27]:
def ReLU(Z):
    return np.maximum(Z, 0)

def ReLU_deriv(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)

def init_params():
    W1 = np.random.randn(128, 784) * np.sqrt(2.0 / 784)
    b1 = np.zeros((128, 1))
    W2 = np.random.randn(10, 128) * np.sqrt(2.0 / 128)
    b2 = np.zeros((10, 1))

    return W1, b1, W2, b2

def forward_prop(x, W1, b1, W2, b2):
    z1 = W1.dot(x) + b1
    a1 = ReLU(z1)
    z2 = W2.dot(a1) + b2
    a2 = softmax(z2)

    return z1, a1, z2, a2

def one_hot_encode(y):
    one_hot_y = np.zeros((y.size, 10))
    one_hot_y[np.arange(y.size), y] = 1
    one_hot_y = one_hot_y
    return one_hot_y.T

def back_prop(z1, a1, a2, W2, X, y):
    m = X.shape[1]
    one_hot_y = one_hot_encode(y)
    dz2 = a2 - one_hot_y
    dW2 = (1 / m) * dz2.dot(a1.T)
    db2 = (1 / m) * a2 - y
    dz1 = W2.T.dot(dz2) * ReLU_deriv(z1)
    dW1 = (1 / m) * dz1.dot(X.T)
    db1 = (1 / m) * np.sum(dz1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2

    return W1, b1, W2, b2

def get_predictions(a2):
    return np.argmax(a2, axis=0)

def get_accuracy(predictions, y):
    return np.sum(predictions == y) / y.size

def mlp_train(x, y, iterations, alpha):
    W1, b1, W2, b2 = init_params()

    for i in range(iterations):
        z1, a1, z2, a2 = forward_prop(x, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = back_prop(z1, a1, a2, W2, x, y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        if i % 10 == 0 or i == iterations - 1:
            predictions = get_predictions(a2)
            acc = get_accuracy(predictions, y)
            print(f"Iteration {i} | acc: {acc}")

    return W1, b1, W2, b2

In [28]:
# 3 layers - 1st input (28 x 28) - 2nd hidden (128) - 3rd output (10)
W1, b1, W2, b2 = mlp_train(X_train, y_train, iterations=200, alpha=0.5)

Iteration 0 | acc: 0.09445
Iteration 10 | acc: 0.7686666666666667
Iteration 20 | acc: 0.80625
Iteration 30 | acc: 0.8370333333333333
Iteration 40 | acc: 0.8920333333333333
Iteration 50 | acc: 0.9017666666666667
Iteration 60 | acc: 0.9067166666666666
Iteration 70 | acc: 0.9098166666666667
Iteration 80 | acc: 0.9113166666666667
Iteration 90 | acc: 0.9171
Iteration 100 | acc: 0.9222
Iteration 110 | acc: 0.9249
Iteration 120 | acc: 0.9275833333333333
Iteration 130 | acc: 0.9301166666666667
Iteration 140 | acc: 0.9323166666666667
Iteration 150 | acc: 0.9345833333333333
Iteration 160 | acc: 0.9361166666666667
Iteration 170 | acc: 0.9376
Iteration 180 | acc: 0.9394666666666667
Iteration 190 | acc: 0.9409666666666666
Iteration 199 | acc: 0.94225
